# Slightly stronger pallor on anemic nails

Color edit for every anemic crop: train, validation, and test. Normal crops are left unchanged.

Reads `Leve` and `Moderada` from `/content/drive/MyDrive/unas_cropped` and writes copies to `/content/drive/MyDrive/anemia/synthetic_anemic_signal`.

The edit is a small pallor shift, stronger for Moderada than Leve:

| Class | Lightness | Red–green (a*) | Saturation |
|-------|-----------|----------------|------------|
| Leve | +6 | −10 | ×0.96 |
| Moderada | +10 | −16 | ×0.92 |

OpenCV LAB uses 0–255, with 128 as neutral a*. Lowering a* removes redness. Originals are not overwritten.

266 anemic crops: 194 Leve, 72 Moderada.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/unas_cropped"
OUT_DIR = "/content/drive/MyDrive/anemia/synthetic_anemic_signal"

# OpenCV LAB / HSV units. a_add < 0 removes redness.
SHIFTS = {
    "Leve": {"l_add": 6.0, "a_add": -10.0, "sat_scale": 0.96},
    "Moderada": {"l_add": 10.0, "a_add": -16.0, "sat_scale": 0.92},
}


In [ ]:
import csv
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
CLASS_FILES = {}
for class_name in ("Leve", "Moderada"):
    src_dir = os.path.join(DATA_DIR, class_name)
    files = []
    for fname in sorted(os.listdir(src_dir)):
        if os.path.splitext(fname)[1].lower() in IMAGE_EXTS:
            files.append(fname)
    CLASS_FILES[class_name] = files
    print(class_name, len(files))

os.makedirs(OUT_DIR, exist_ok=True)


def add_pallor(bgr, l_add, a_add, sat_scale):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    lab[:, :, 0] = np.clip(lab[:, :, 0] + l_add, 0, 255)
    lab[:, :, 1] = np.clip(lab[:, :, 1] + a_add, 0, 255)
    shifted = cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2BGR)
    hsv = cv2.cvtColor(shifted, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * sat_scale, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)


rows = []
preview = []
missing = []
for class_name, files in CLASS_FILES.items():
    src_dir = os.path.join(DATA_DIR, class_name)
    dst_dir = os.path.join(OUT_DIR, class_name)
    os.makedirs(dst_dir, exist_ok=True)
    shift = SHIFTS[class_name]
    for fname in files:
        image_id = os.path.splitext(fname)[0]
        src = os.path.join(src_dir, fname)
        if not os.path.isfile(src):
            missing.append(src)
            continue
        bgr = cv2.imread(src)
        if bgr is None:
            missing.append(src)
            continue
        out = add_pallor(bgr, **shift)
        dst = os.path.join(dst_dir, fname)
        cv2.imwrite(dst, out)
        rows.append({
            "image_id": image_id,
            "class_name": class_name,
            "src": src,
            "dst": dst,
            **shift,
        })
        if len(preview) < 4:
            preview.append((image_id, class_name, bgr, out))

manifest = os.path.join(OUT_DIR, "manifest.csv")
with open(manifest, "w", newline="", encoding="utf-8") as f:
    fields = ["image_id", "class_name", "src", "dst", "l_add", "a_add", "sat_scale"]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)

print("wrote", len(rows), "images to", OUT_DIR)
print("manifest", manifest)
if missing:
    print("missing", len(missing))
    for path in missing[:10]:
        print(" ", path)

fig, axes = plt.subplots(len(preview), 2, figsize=(6, 3 * max(len(preview), 1)))
if len(preview) == 1:
    axes = np.array([axes])
for row, (image_id, class_name, bgr, out) in enumerate(preview):
    axes[row, 0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f"{image_id} {class_name} original")
    axes[row, 1].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
    axes[row, 1].set_title("more pallor")
    axes[row, 0].axis("off")
    axes[row, 1].axis("off")
plt.tight_layout()
plt.show()
